# CryptoStatArb — Research Notebook

**Statistical Arbitrage in Cryptocurrencies** (WSQ Course Project)

Goal: research profitable **momentum and/or reversal** strategies on crypto,
backtest unconstrained, apply realistic execution costs (Binance.US, Tier 1),
and evaluate returns / vol / Sharpe / max drawdown / alpha-beta.

Exchange: [Binance.US](https://www.binance.us/fees)

In [1]:
# --- Core libs ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
%matplotlib inline

# Canonical dataset — build once via download_data.ipynb, then load here:
DATA_DIR = Path('data')
# panel = pd.read_pickle(DATA_DIR / 'binance_us_hourly_ohlcv.pk')
# close, volume = panel['Close'], panel['Volume']
# ret = close.pct_change()

In [2]:
# --- Binance.US commission (published standard rates, checked 2026-08-27) ---
FEES = {
    'maker_bps': 0.0,      # 0.00%  limit orders that ADD liquidity
    'taker_bps': 2.0,      # 0.02%  market orders that TAKE liquidity
    'bnb_discount': 0.05,  # 5% off maker & taker when fees paid in BNB
}

# Course-project baseline (from ClassProject brief) for cross-checking backtests:
#   market order = 7 bps commission + 13 bps assumed slippage = 20 bps all-in
#   limit order  = 7 bps commission only
PROJECT_COST_ASSUMPTION = {
    'commission_bps': 7.0,
    'slippage_bps': 13.0,
    'market_all_in_bps': 20.0,
    'limit_bps': 7.0,
}

# Asset-dependent costs (half-spread, impact) are filled in per symbol later, e.g.:
#   SYMBOL_TCOSTS = {'BTCUSDT': {'half_spread_bps': ..., 'impact_bps': ...}, ...}
SYMBOL_TCOSTS = {}

def bps(x):
    """basis points -> decimal (e.g. 20 -> 0.0020)."""
    return x / 1e4

FEES

{'maker_bps': 0.0, 'taker_bps': 2.0, 'bnb_discount': 0.05}

## Performance & Risk Metrics (reusable helpers)

Conventions match the WSQ lectures. Inputs:
- `strat_ret` — per-bar strategy **return** Series.
- `port` — the **weights** DataFrame (time × asset) used for turnover.
- `benchmark` — a market return Series (e.g. equal-weight universe) for beta.
- `ann` — periods per year for annualization: hourly `24*365`; b-hour `24*365/b`.

In [3]:
# --- Performance & risk metrics ---------------------------------------------
ANN_HOURLY = 24 * 365   # bars per year for hourly data (crypto trades 24/7)

def sharpe(ret, ann=ANN_HOURLY):
    """Annualized Sharpe ratio of a per-period return series."""
    return ret.mean() / ret.std() * np.sqrt(ann)

def drawdown(equity):
    """Drawdown series from an equity/price curve (WSQ convention)."""
    return equity / equity.expanding(min_periods=1).max() - 1

def equity_curve(ret):
    """Compounded equity from per-period returns (starts at 1)."""
    return (1 + ret.fillna(0)).cumprod()

def max_drawdown(ret):
    """Worst peak-to-trough drawdown (a negative number)."""
    return drawdown(equity_curve(ret)).min()

def max_dd_duration(ret):
    """Longest underwater stretch in BARS (peak until a new high is made)."""
    eq = equity_curve(ret)
    underwater = eq < eq.expanding(min_periods=1).max()
    return int(underwater.groupby((~underwater).cumsum()).cumsum().max())

def turnover(port):
    """Per-period turnover: sum_i |w_it - w_i,t-1|  (WSQ formula). Returns a Series."""
    return (port.fillna(0) - port.shift().fillna(0)).abs().sum(1)

def beta(strat_ret, benchmark_ret):
    """Market beta = OLS slope of strategy returns on a benchmark return series."""
    d = pd.concat([strat_ret, benchmark_ret], axis=1).dropna()
    X = sm.add_constant(d.iloc[:, 1])
    return sm.OLS(d.iloc[:, 0], X).fit().params.iloc[1]

def beta_contribution(strat_ret, benchmark_ret):
    """The beta * benchmark component of returns (the 'market' part of P&L)."""
    return beta(strat_ret, benchmark_ret) * benchmark_ret

def performance_summary(strat_ret, port=None, benchmark=None, ann=ANN_HOURLY, name='strategy'):
    """One-row summary of all key metrics for a strategy return series."""
    s = {
        'ann_return':      strat_ret.mean() * ann,
        'ann_vol':         strat_ret.std() * np.sqrt(ann),
        'sharpe':          sharpe(strat_ret, ann),
        'avg_turnover':    turnover(port).mean() if port is not None else np.nan,
        'max_drawdown':    max_drawdown(strat_ret),
        'max_dd_dur_bars': max_dd_duration(strat_ret),
        'beta':            beta(strat_ret, benchmark) if benchmark is not None else np.nan,
    }
    return pd.Series(s, name=name)

def summary_table(strats, ports=None, benchmark=None, ann=ANN_HOURLY):
    """Metrics table (rows = strategies) from a dict {name: return_series}.
       `ports`: optional {name: weights_df};  `ann`: scalar or {name: ann}."""
    rows = []
    for nm, r in strats.items():
        p = ports.get(nm) if isinstance(ports, dict) else None
        a = ann.get(nm, ANN_HOURLY) if isinstance(ann, dict) else ann
        rows.append(performance_summary(r, p, benchmark, a, nm))
    return pd.DataFrame(rows)

print('metrics ready:', [f.__name__ for f in
      (sharpe, drawdown, max_drawdown, max_dd_duration, turnover, beta, performance_summary, summary_table)])

metrics ready: ['sharpe', 'drawdown', 'max_drawdown', 'max_dd_duration', 'turnover', 'beta', 'performance_summary', 'summary_table']


### Using the metrics

```python
mkt = ret.mean(axis=1)                       # equal-weight 'market' benchmark

# one strategy:
performance_summary(strat_ret, port, benchmark=mkt, ann=24*365)

# several at once (e.g. per holding interval) -> a table:
summary_table({'1h': r1, '2h': r2},
              ports={'1h': p1, '2h': p2},
              benchmark=mkt,
              ann={'1h': 24*365, '2h': 24*365/2})
```
For a b-hour strategy set `ann = 24*365/b`. `max_dd_dur_bars` is in bars — multiply by the
bar length for wall-clock time.